In [ ]:
import torch
import torch.nn as nn


In [ ]:
# torch.manual_seed(42)
inputs = torch.tensor([1.0,2.0,3.0])

In [ ]:
neuron = nn.Linear(in_features=3, out_features=1)

In [ ]:
output = neuron(inputs)


In [ ]:
output

tensor([1.3077], grad_fn=<ViewBackward0>)

In [ ]:
# verify
neuron.bias

Parameter containing:
tensor([0.4087], requires_grad=True)

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [5]:
# downlaod training data from open dataset
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ]),
)

# download Testing data
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ])
)

100%|██████████| 26.4M/26.4M [00:00<00:00, 114MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.62MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 55.0MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 28.4MB/s]


In [10]:
batch_size =  64
# create data loader
train_dataloader = DataLoader(training_data , batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)
for X ,y in test_dataloader :
  print(f"shape of x[N,C,H,W]:{X.shape}")
  print(f"shape of y: {y.shape} {y.dtype}")
  break



shape of x[N,C,H,W]:torch.Size([64, 1, 28, 28])
shape of y: torch.Size([64]) torch.int64


In [14]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"using {device} device ")

using cuda device 


In [ ]:
# define model
class NeuralNetwork(nn.Module):
  def __init__(self):
    super()._init_()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28,512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )
def forward(self, X):
  X =self.flateen(X)
  logits = self.linear_relu_stack(x)
  return logits



In [16]:
# define model
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28,512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )

  def forward(self, X):
    X = self.flatten(X)
    logits = self.linear_relu_stack(X)
    return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [17]:
# optimizing the model parameter
loss_fn =nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr =1e-3)

In [21]:
def train(dataloader, model ,loss_fn , optimizer):
  size =len(dataloader.dataset)
  model.train()
  for batch, (X,y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    # compute prediction error
    pred = model(X)
    loss = loss_fn(pred, y )

    # Backpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()


    if batch % 100 == 0:
       loss, current =loss.item(), (batch+1)*len(X)
       print(f"loass:{loss:>7f}  [{current:>5d}/{size:>5d}]")

In [22]:
# We also check the model’s performance against the test dataset to ensure it is learning

def test(dataloader,model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct =0,0
  with torch.no_grad():
    for X, y in dataloader:
      X, y  = X.to(device), y.to(device)
      pred =model(X)
      test_loss += loss_fn(pred,y ).item()
      correct +=(pred.argmax(1) == y ).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"test_error: \n Acuuracy : {(100*correct):>0.1f}%, Avg loss:{test_loss:>8f}\n")

The training process is conducted over several iterations (epochs). During each epoch, the model learns parameters to make better predictions. We print the model’s accuracy and loss at each epoch; we’d like to see the accuracy increase and the loss decrease with every epoch.

In [23]:
epochs =  5
for t in range(epochs):
  print(f"EPoch {t+1}\n -------------")
  train(train_dataloader, model, loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)
print("Done!")


EPoch 1
 -------------
loass:2.312126  [   64/60000]
loass:2.300127  [ 6464/60000]
loass:2.285123  [12864/60000]
loass:2.268302  [19264/60000]
loass:2.262496  [25664/60000]
loass:2.226212  [32064/60000]
loass:2.238428  [38464/60000]
loass:2.211404  [44864/60000]
loass:2.203543  [51264/60000]
loass:2.170936  [57664/60000]
test_error: 
 Acuuracy : 28.7%, Avg loss:2.170086

EPoch 2
 -------------
loass:2.185040  [   64/60000]
loass:2.173453  [ 6464/60000]
loass:2.127688  [12864/60000]
loass:2.130050  [19264/60000]
loass:2.092682  [25664/60000]
loass:2.032037  [32064/60000]
loass:2.051300  [38464/60000]
loass:1.992844  [44864/60000]
loass:1.988604  [51264/60000]
loass:1.913828  [57664/60000]
test_error: 
 Acuuracy : 60.1%, Avg loss:1.919304

EPoch 3
 -------------
loass:1.955791  [   64/60000]
loass:1.923220  [ 6464/60000]
loass:1.823047  [12864/60000]
loass:1.843028  [19264/60000]
loass:1.744666  [25664/60000]
loass:1.697134  [32064/60000]
loass:1.700571  [38464/60000]
loass:1.626920  [44

In [24]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [25]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))


<All keys matched successfully>

In [26]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
